<a href="https://colab.research.google.com/github/afrineejhilik71-arch/plant-disease-detection/blob/main/Tomato_experiment_run01b_baseline_top5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tomato experiment template (Agri-Vision Bangladesh, all 5 classes, fold 0)
One notebook for every controlled experiment. **Change only the settings in section 2**, then Runtime → Run all.

**Selection rule:** models and settings are compared on the **validation** fold (macro-F1). Fold 0's test results are reported but treated as *development* data, because fold 0 has already been used across several runs. Untouched folds 1–4 are kept for the final evaluation.
One self-contained notebook: data → cleaning → split → train 8 pretrained CNNs → results.
It needs only `MyDrive/thesis/data/raw/agri_vision.zip`. Nothing from earlier notebooks is required.

**Steps:** 1 GPU · 2 settings + Drive · 3 unzip · 4 audit + clean · 5 folds · 6 training script · 8 train · 9 results · 10 compare runs (validation)

**Training protocol (v3, fixed):** head warm-up up to 15 epochs → full fine-tuning with learning-rate warm-up (0 → 3e-5, 2 epochs) and cosine decay, early stopping on validation accuracy (patience 10), best weights restored. Class weights, train-only augmentation, 224×224 with padding.

Runtime: **T4 GPU**. After a disconnect: Runtime → Run all (finished models are skipped).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Experiment settings (the only cell you change) + Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imagehash
from pathlib import Path
import json, time, gc, hashlib
import numpy as np, pandas as pd, psutil, imagehash
from PIL import Image, ImageOps
from tqdm.auto import tqdm

PROJECT  = Path('/content/drive/MyDrive/thesis')
ZIP_PATH = PROJECT / 'data/raw/agri_vision.zip'
# ===================== EXPERIMENT SETTINGS =====================
RUN_NAME        = 'run01b_baseline_top5'   # a new name for every run
LABEL_SMOOTHING = 0.0                      # Run 03: 0.1
IMG_SIZE        = 224
MODEL_NAMES     = ['VGG16', 'ConvNeXtTiny', 'EfficientNetV2B0', 'DenseNet121', 'InceptionV3']   # top 5 of Run 01
NOTE            = 'Reference for Stage 1: Run 01 protocol, top 5 models, with validation macro-F1 recorded'
# ===============================================================
OUT      = PROJECT / 'experiments' / 'tomato' / RUN_NAME
for sub in ['history', 'preds']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT / 'results.csv'

CROP, FOLD, SEED, N_FOLDS = 'Tomato', 0, 42, 5
print('Results go to', OUT)

Mounted at /content/drive
Results go to /content/drive/MyDrive/thesis/experiments/tomato/run01b_baseline_top5


## 3. Unzip the images to local disk (about 1 minute)

In [ ]:
UNZIP_DIR = Path('/content/data/agri_vision')
if not UNZIP_DIR.exists():
    UNZIP_DIR.mkdir(parents=True)
    !unzip -q "{ZIP_PATH}" -d "{UNZIP_DIR}"
    for inner in list(UNZIP_DIR.rglob('*.zip')):
        !unzip -q -o "{inner}" -d "{inner.parent}"
ORIG_DIR = [p for p in UNZIP_DIR.rglob('*') if p.is_dir() and 'original' in p.name.lower()][0]
print('Original images folder:', ORIG_DIR)

Original images folder: /content/data/agri_vision/Agri-Vision4 A Comprehensive Multi-Crop Leaf Disea/Bottle Gourd, Papaya, Tomato, and Zucchini Original Dataset


## 4. Audit and clean (same rules as before, about 1 minute)
All crops are hashed so the folds are identical to the earlier runs; only Tomato is trained afterwards.
Rules: remove unreadable files → remove identical images with different labels → keep one copy of identical images → remove near-identical images with different labels → exclude Zucchini.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
rows = []
for p in tqdm(sorted(q for q in ORIG_DIR.rglob('*') if q.suffix.lower() in IMG_EXT), desc='Hashing'):
    parts = p.relative_to(ORIG_DIR).parts
    rec = {'rel_path': '/'.join(parts), 'split': parts[0], 'label_raw': parts[-2]}
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            rec['phash'] = str(imagehash.phash(im.convert('RGB')))
        rec['md5'] = hashlib.md5(p.read_bytes()).hexdigest()
        rec['corrupt'] = False
    except Exception:
        rec['corrupt'] = True
    rows.append(rec)
df = pd.DataFrame(rows).sort_values('rel_path').reset_index(drop=True)
df['label'] = df.label_raw.replace({'Tomato_Downy Mildew': 'Tomato_Downy_Mildew', 'Zucchini_Downy_Midew': 'Zucchini_Downy_Mildew'})
df['crop'] = df.label.str.extract(r'^(Bottle_Gourd|Papaya|Tomato|Zucchini)', expand=False)
df['keep'], df['reason'] = True, ''

def drop(mask, reason):
    m = mask & df.keep
    df.loc[m, 'keep'], df.loc[m, 'reason'] = False, reason
    print(f'{reason:24s} removed {int(m.sum()):5d}   remaining {int(df.keep.sum()):5d}')

drop(df.corrupt.astype(bool), 'unreadable')
g = df[df.keep].groupby('md5').label.nunique();   drop(df.md5.isin(g[g > 1].index), 'label_conflict_exact')
drop(df.keep & df.duplicated('md5'), 'duplicate_copy')
g = df[df.keep].groupby('phash').label.nunique(); drop(df.phash.isin(g[g > 1].index), 'label_conflict_near')
drop(df.crop.eq('Zucchini'), 'excluded_zucchini')

Hashing:   0%|          | 0/5246 [00:00<?, ?it/s]

unreadable               removed     0   remaining  5246
label_conflict_exact     removed   754   remaining  4492
duplicate_copy           removed   463   remaining  4029
label_conflict_near      removed    18   remaining  4011
excluded_zucchini        removed   262   remaining  3749


## 5. Five stratified folds, grouped by near-duplicates; select Tomato

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
kept = df[df.keep].copy()
kept['fold'] = -1
for k, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
                            .split(kept, kept.label, groups=kept.phash)):
    kept.iloc[te, kept.columns.get_loc('fold')] = k
assert kept.md5.is_unique and kept.groupby('phash').fold.nunique().max() == 1

d = kept[kept.crop == CROP].sort_values('rel_path').reset_index(drop=True)
CLASSES = sorted(d.label.unique())
y = d.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
te, va = (d.fold == FOLD).to_numpy(), (d.fold == (FOLD + 1) % N_FOLDS).to_numpy()
tr = ~(te | va)
d.to_csv(OUT / 'tomato_split.csv', index=False)
print(f'{CROP}: {len(d)} clean images, {len(CLASSES)} classes')
display(pd.crosstab(d.label, np.select([te, va], ['test', 'validation'], 'train')))

Tomato: 795 clean images, 5 classes


col_0,test,train,validation
label,,,
Tomato_Downy_Mildew,8,36,11
Tomato_Healthy,49,151,46
Tomato_Mosaic,37,113,41
Tomato_Spot,43,152,43
Tomato_White_Spot,11,41,13


## 6. Training settings and the training script
Each model is trained in its **own separate process**. When the process ends, all of its memory is released, so the RAM cannot fill up across models (this caused the crash before).

In [ ]:
cfg = {'run': RUN_NAME, 'note': NOTE, 'crop': CROP, 'fold': FOLD, 'seed': SEED, 'n_folds': N_FOLDS,
       'img_size': IMG_SIZE, 'batch': 32, 'label_smoothing': LABEL_SMOOTHING, 'models': MODEL_NAMES, 'head_epochs': 15, 'head_lr': 1e-3,
       'ft_epochs': 40, 'ft_lr': 3e-5, 'warmup_epochs': 2, 'ft_patience': 10,
       'optimizer': 'AdamW(weight_decay=1e-4, clipnorm=1.0)',
       'early_stopping': 'head: val_loss p3; fine-tune: val_accuracy p10; restore best',
       'augmentation': 'flip h+v, rotation 0.1, zoom 0.1, contrast 0.1 (train only)',
       'orig_dir': str(ORIG_DIR), 'out': str(OUT)}
(OUT / 'config.json').write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

{
  "run": "run01b_baseline_top5",
  "note": "Reference for Stage 1: Run 01 protocol, top 5 models, with validation macro-F1 recorded",
  "crop": "Tomato",
  "fold": 0,
  "seed": 42,
  "n_folds": 5,
  "img_size": 224,
  "batch": 32,
  "label_smoothing": 0.0,
  "models": [
    "VGG16",
    "ConvNeXtTiny",
    "EfficientNetV2B0",
    "DenseNet121",
    "InceptionV3"
  ],
  "head_epochs": 15,
  "head_lr": 0.001,
  "ft_epochs": 40,
  "ft_lr": 3e-05,
  "warmup_epochs": 2,
  "ft_patience": 10,
  "optimizer": "AdamW(weight_decay=1e-4, clipnorm=1.0)",
  "early_stopping": "head: val_loss p3; fine-tune: val_accuracy p10; restore best",
  "augmentation": "flip h+v, rotation 0.1, zoom 0.1, contrast 0.1 (train only)",
  "orig_dir": "/content/data/agri_vision/Agri-Vision4 A Comprehensive Multi-Crop Leaf Disea/Bottle Gourd, Papaya, Tomato, and Zucchini Original Dataset",
  "out": "/content/drive/MyDrive/thesis/experiments/tomato/run01b_baseline_top5"
}


In [ ]:
%%writefile /content/train_one.py
# Trains ONE model on the split saved by the notebook, then appends its results. Usage: python train_one.py MODEL
import os, sys, json, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf, keras
from keras import layers
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

name = sys.argv[1]
OUT = Path(sys.argv[2]); cfg = json.loads((OUT / 'config.json').read_text())
ORIG_DIR, IMG, BATCH, SEED, FOLD = Path(cfg['orig_dir']), cfg['img_size'], cfg['batch'], cfg['seed'], cfg['fold']
d = pd.read_csv(OUT / 'tomato_split.csv')
CLASSES = sorted(d.label.unique())
y = d.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
te = (d.fold == FOLD).to_numpy(); va = (d.fold == (FOLD + 1) % cfg['n_folds']).to_numpy(); tr = ~(te | va)

A, identity = keras.applications, (lambda x: x)
MODELS = {'VGG16': (A.VGG16, A.vgg16.preprocess_input), 'ResNet50': (A.ResNet50, A.resnet50.preprocess_input),
          'DenseNet121': (A.DenseNet121, A.densenet.preprocess_input),
          'InceptionV3': (A.InceptionV3, A.inception_v3.preprocess_input),
          'Xception': (A.Xception, A.xception.preprocess_input),
          'MobileNetV3Large': (A.MobileNetV3Large, identity),
          'EfficientNetV2B0': (A.EfficientNetV2B0, identity), 'ConvNeXtTiny': (A.ConvNeXtTiny, identity)}
keras.mixed_precision.set_global_policy('float32' if name in {'MobileNetV3Large', 'Xception'} else 'mixed_float16')
keras.utils.set_random_seed(SEED + FOLD)

X = np.stack([np.asarray(ImageOps.pad(Image.open(ORIG_DIR / p).convert('RGB'), (IMG, IMG), color=(0, 0, 0)),
                         dtype=np.uint8) for p in d.rel_path])
augment = keras.Sequential([layers.RandomFlip('horizontal_and_vertical', dtype='float32'),
                            layers.RandomRotation(0.1, dtype='float32'), layers.RandomZoom(0.1, dtype='float32'),
                            layers.RandomContrast(0.1, dtype='float32')])
ctor, prep = MODELS[name]

def make_ds(mask, train):
    ds = tf.data.Dataset.from_tensor_slices((X[mask], y[mask]))
    if train:
        ds = ds.shuffle(int(mask.sum()), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), tf.one_hot(b, len(CLASSES))),
                             num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.map(lambda a, b: (prep(a), b), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

LOSS = keras.losses.CategoricalCrossentropy(label_smoothing=cfg['label_smoothing'])
base = ctor(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3)); base.trainable = False
inp = keras.Input((IMG, IMG, 3))
x = layers.GlobalAveragePooling2D()(base(inp, training=False))
model = keras.Model(inp, layers.Dense(len(CLASSES), activation='softmax', dtype='float32')(layers.Dropout(0.3)(x)))
ds_tr, ds_va, ds_te = make_ds(tr, True), make_ds(va, False), make_ds(te, False)
cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y[tr])))
opt = lambda lr: keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4, clipnorm=1.0)

t0 = time.perf_counter()
model.compile(opt(cfg['head_lr']), LOSS, metrics=['accuracy'])
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=cfg['head_epochs'], class_weight=cw, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=3, restore_best_weights=True)])
base.trainable = True
steps = int(np.ceil(tr.sum() / BATCH))
sched = keras.optimizers.schedules.CosineDecay(0.0, (cfg['ft_epochs'] - cfg['warmup_epochs']) * steps,
                                               warmup_target=cfg['ft_lr'], warmup_steps=cfg['warmup_epochs'] * steps)
model.compile(opt(sched), LOSS, metrics=['accuracy'])
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=cfg['ft_epochs'], class_weight=cw, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping('val_accuracy', mode='max', patience=cfg['ft_patience'],
                                                        restore_best_weights=True),
                          keras.callbacks.TerminateOnNaN()])
minutes = (time.perf_counter() - t0) / 60
pred = model.predict(ds_te, verbose=0).argmax(1); true = y[te]
vpred = model.predict(ds_va, verbose=0).argmax(1); vtrue = y[va]
LAB = range(len(CLASSES))
per = lambda fn, t, p: json.dumps(dict(zip(CLASSES, np.round(fn(t, p, average=None, labels=LAB, zero_division=0), 3).tolist())))

(OUT / 'history').mkdir(exist_ok=True); (OUT / 'preds').mkdir(exist_ok=True)
pd.concat([pd.DataFrame(h1.history).assign(phase='head'), pd.DataFrame(h2.history).assign(phase='finetune')]
          ).to_csv(OUT / 'history' / f'{name}.csv', index=False)
pd.DataFrame({'rel_path': d.rel_path[te].values, 'y_true': true, 'y_pred': pred}).to_csv(OUT / 'preds' / f'{name}.csv', index=False)
best = int(np.nanargmax(h2.history['val_accuracy']))
row = {'run': cfg['run'], 'crop': cfg['crop'], 'model': name, 'fold': FOLD, 'n_train': int(tr.sum()), 'n_val': int(va.sum()), 'n_test': int(te.sum()),
       'img_size': IMG, 'label_smoothing': cfg['label_smoothing'],
       'val_accuracy': accuracy_score(vtrue, vpred), 'val_macro_f1': f1_score(vtrue, vpred, average='macro', zero_division=0),
       'accuracy': accuracy_score(true, pred), 'balanced_accuracy': balanced_accuracy_score(true, pred),
       'macro_precision': precision_score(true, pred, average='macro', zero_division=0),
       'macro_recall': recall_score(true, pred, average='macro', zero_division=0),
       'macro_f1': f1_score(true, pred, average='macro', zero_division=0),
       'weighted_f1': f1_score(true, pred, average='weighted', zero_division=0),
       'cohen_kappa': cohen_kappa_score(true, pred), 'mcc': matthews_corrcoef(true, pred),
       'best_val_acc': float(h2.history['val_accuracy'][best]), 'train_acc_at_best': float(h2.history['accuracy'][best]),
       'epochs_head': len(h1.history['loss']), 'epochs_finetune': len(h2.history['loss']),
       'params_M': model.count_params() / 1e6, 'train_minutes': minutes,
       'per_class_precision': per(precision_score, true, pred), 'per_class_recall': per(recall_score, true, pred),
       'per_class_f1': per(f1_score, true, pred), 'val_per_class_f1': per(f1_score, vtrue, vpred),
       'confusion_matrix': json.dumps(confusion_matrix(true, pred, labels=range(len(CLASSES))).tolist())}
res = OUT / 'results.csv'
new = pd.DataFrame([row])
if res.exists():
    new = pd.concat([pd.read_csv(res), new], ignore_index=True)
new.to_csv(res, index=False)
print(f"{name:17s} VAL macroF1 {row['val_macro_f1']:.3f} | test acc {row['accuracy']:.3f} macroF1 {row['macro_f1']:.3f} | "
      f"train_acc {row['train_acc_at_best']:.3f} ({minutes:.1f} min)")

Writing /content/train_one.py


## 8. Train — one separate process per model (about 40–60 minutes for 5 models at 224 px)

In [ ]:
import subprocess
done = set(pd.read_csv(RESULTS_CSV).model) if RESULTS_CSV.exists() else set()
todo = [m for m in MODEL_NAMES if m not in done]
print(f'{len(done)} finished, {len(todo)} to do')
for i, name in enumerate(todo, 1):
    p = subprocess.run(['python', '/content/train_one.py', name, str(OUT)], capture_output=True, text=True)
    last = [l for l in p.stdout.strip().splitlines() if l.strip()]
    if p.returncode == 0 and last:
        print(f'[{i}/{len(todo)}] {last[-1]}   | RAM now {psutil.virtual_memory().used/1e9:.1f} GB')
    else:
        print(f'[{i}/{len(todo)}] {name} FAILED:\n' + p.stderr[-1500:])
print('Done.')

0 finished, 5 to do
[1/5] VGG16             VAL macroF1 0.720 | test acc 0.818 macroF1 0.762 | train_acc 0.927 (7.7 min)   | RAM now 1.5 GB
[2/5] ConvNeXtTiny      VAL macroF1 0.738 | test acc 0.743 macroF1 0.709 | train_acc 0.937 (10.4 min)   | RAM now 1.5 GB
[3/5] EfficientNetV2B0  VAL macroF1 0.643 | test acc 0.703 macroF1 0.651 | train_acc 0.732 (12.8 min)   | RAM now 1.9 GB
[4/5] DenseNet121       VAL macroF1 0.710 | test acc 0.784 macroF1 0.749 | train_acc 0.882 (22.2 min)   | RAM now 1.9 GB
[5/5] InceptionV3       VAL macroF1 0.641 | test acc 0.730 macroF1 0.681 | train_acc 0.876 (10.9 min)   | RAM now 2.0 GB
Done.


## 9. Results: validation (for selection) and test (development, reported)

In [ ]:
r = pd.read_csv(RESULTS_CSV).drop_duplicates('model', keep='last').sort_values('val_macro_f1', ascending=False)
pct = lambda df, c: (100 * df[c]).round(2)
val = pd.DataFrame({'model': r.model, 'VAL macro-F1': pct(r, 'val_macro_f1'), 'VAL accuracy': pct(r, 'val_accuracy'),
                    'train acc at best': pct(r, 'train_acc_at_best'), 'epochs fine-tune': r.epochs_finetune})
test = pd.DataFrame({'model': r.model, 'accuracy': pct(r, 'accuracy'), 'macro precision': pct(r, 'macro_precision'),
                     'macro recall': pct(r, 'macro_recall'), 'macro-F1': pct(r, 'macro_f1'),
                     'balanced acc.': pct(r, 'balanced_accuracy'), 'MCC': pct(r, 'mcc')})
print(f'{RUN_NAME} | VALIDATION fold ({int(r.n_val.iloc[0])} images): used to compare models and settings'); display(val)
print(f'\n{RUN_NAME} | TEST fold 0 ({int(r.n_test.iloc[0])} images): development results, reported only'); display(test)

pc = {k: pd.DataFrame({m: json.loads(s) for m, s in zip(r.model, r[col])}).T * 100
      for k, col in [('precision', 'per_class_precision'), ('recall', 'per_class_recall'), ('F1', 'per_class_f1')]}
for k, t in pc.items():
    print(f'\nTest per-class {k} (%):'); display(t.round(1))

with pd.ExcelWriter(OUT / f'{RUN_NAME}_results.xlsx') as xw:
    val.to_excel(xw, sheet_name='validation', index=False)
    test.to_excel(xw, sheet_name='test_fold0', index=False)
    for k, t in pc.items():
        t.round(1).to_excel(xw, sheet_name=f'per_class_{k}')
    r.drop(columns=['confusion_matrix']).to_excel(xw, sheet_name='all_columns', index=False)
    pd.Series(cfg).astype(str).to_frame('value').to_excel(xw, sheet_name='protocol')
print('Saved', OUT / f'{RUN_NAME}_results.xlsx')

run01b_baseline_top5 | VALIDATION fold (154 images): used to compare models and settings


,model,VAL macro-F1,VAL accuracy,train acc at best,epochs fine-tune
1,ConvNeXtTiny,73.78,78.57,93.71,37
0,VGG16,71.98,77.92,92.70,34
3,DenseNet121,71.00,77.92,88.24,34
2,EfficientNetV2B0,64.29,70.13,73.23,36
4,InceptionV3,64.14,72.08,87.63,30



run01b_baseline_top5 | TEST fold 0 (148 images): development results, reported only


,model,accuracy,macro precision,macro recall,macro-F1,balanced acc.,MCC
1,ConvNeXtTiny,74.32,70.46,72.49,70.89,72.49,65.16
0,VGG16,81.76,78.65,75.78,76.23,75.78,75.60
3,DenseNet121,78.38,74.65,75.43,74.90,75.43,70.58
2,EfficientNetV2B0,70.27,65.97,66.79,65.14,66.79,60.28
4,InceptionV3,72.97,68.52,69.20,68.05,69.20,63.60



Test per-class precision (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,63.6,85.2,63.6,73.2,66.7
VGG16,66.7,91.8,85.2,74.5,75.0
DenseNet121,77.8,90.2,69.7,75.6,60.0
EfficientNetV2B0,71.4,81.5,68.0,72.1,36.8
InceptionV3,66.7,81.8,69.6,70.0,54.5



Test per-class recall (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,87.5,93.9,56.8,69.8,54.5
VGG16,75.0,91.8,62.2,95.3,54.5
DenseNet121,87.5,93.9,62.2,79.1,54.5
EfficientNetV2B0,62.5,89.8,45.9,72.1,63.6
InceptionV3,75.0,91.8,43.2,81.4,54.5



Test per-class F1 (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,73.7,89.3,60.0,71.4,60.0
VGG16,70.6,91.8,71.9,83.7,63.2
DenseNet121,82.4,92.0,65.7,77.3,57.1
EfficientNetV2B0,66.7,85.4,54.8,72.1,46.7
InceptionV3,70.6,86.5,53.3,75.3,54.5


Saved /content/drive/MyDrive/thesis/experiments/tomato/run01b_baseline_top5/run01b_baseline_top5_results.xlsx


## 10. Compare all runs that recorded validation macro-F1

In [ ]:
runs = []
for f in sorted((PROJECT / 'experiments' / 'tomato').glob('*/results.csv')):
    x = pd.read_csv(f)
    if 'val_macro_f1' in x:
        runs.append(x.drop_duplicates('model', keep='last').assign(run=f.parent.name))
if runs:
    allr = pd.concat(runs)
    piv = (100 * allr.pivot_table(index='model', columns='run', values='val_macro_f1')).round(1)
    piv.loc['MEAN'] = piv.mean().round(1)
    print('VALIDATION macro-F1 (%) per model and run:'); display(piv)
    piv.to_csv(PROJECT / 'experiments' / 'tomato' / 'compare_validation_macro_f1.csv')

VALIDATION macro-F1 (%) per model and run:


run,run01b_baseline_top5
model,
ConvNeXtTiny,73.8
DenseNet121,71.0
EfficientNetV2B0,64.3
InceptionV3,64.1
VGG16,72.0
MEAN,69.0
